
# Credit Risk & Loan Default Analytics

## Project Overview

This project analyzes loan portfolio data to identify credit risk patterns and understand loan default behavior.

## Objectives

- Analyze loan portfolio performance.
- Measure default rates and financial exposure.
- Identify risk patterns across customer profiles.
- Generate business insights using SQL and PySpark.

## Architecture

The project follows the Medallion Architecture:

- Bronze: Raw loan data.
- Silver: Cleaned and validated data.
- Gold: Aggregated credit risk indicators.

## Technologies

- Databricks
- PySpark
- SQL
- Delta Lake
- Databricks Dashboards

## Dataset

Synthetic loan dataset generated for educational and analytical purposes.


In [0]:

from pyspark.sql import functions as F
from pyspark.sql import types as T

from datetime import date

print("Credit Risk Analytics")
print("Spark version:", spark.version)
print("Environment successfully initialized!")


Credit Risk Analytics
Spark version: 4.2.0
Environment successfully initialized!


In [0]:

# Generate synthetic loan portfolio data

from pyspark.sql import functions as F

# Create 10,000 simulated loan records
df = spark.range(10000)

# Generate customer and loan attributes
df = (
    df
    .withColumn("loan_id", F.concat(F.lit("LN"), F.col("id")))
    .withColumn("age", (F.rand(10) * 50 + 21).cast("int"))
    .withColumn("annual_income", (F.rand(20) * 130000 + 20000).cast("int"))
    .withColumn("credit_score", (F.rand(30) * 551 + 300).cast("int"))
    .withColumn("loan_amount", (F.rand(40) * 49000 + 1000).cast("double"))
    .withColumn(
        "loan_purpose",
        F.when(F.rand(50) < 0.25, "Personal")
         .when(F.rand(51) < 0.50, "Education")
         .when(F.rand(52) < 0.75, "Home Improvement")
         .otherwise("Debt Consolidation")
    )
    .withColumn(
        "employment_type",
        F.when(F.rand(60) < 0.40, "Full-time")
         .when(F.rand(61) < 0.70, "Self-employed")
         .otherwise("Part-time")
    )
    .withColumn("debt_to_income", F.round(F.rand(70) * 0.65, 2))
)

# Calculate interest rate based on credit score
df = df.withColumn(
    "interest_rate",
    F.round(
        5 + (850 - F.col("credit_score")) * 0.035,
        2
    )
)

# Simulate default probability
df = df.withColumn(
    "default_probability",
    F.when(F.col("credit_score") < 580, 0.25)
     .when(F.col("credit_score") < 670, 0.14)
     .otherwise(0.04)
    + F.when(F.col("debt_to_income") > 0.40, 0.12).otherwise(0.0)
)

# Generate default indicator
df = df.withColumn(
    "default_flag",
    F.when(
        F.rand(80) < F.col("default_probability"), 1
    ).otherwise(0)
)

# Define loan status
df = df.withColumn(
    "loan_status",
    F.when(F.col("default_flag") == 1, "Default")
     .otherwise("Non-Default")
)

# Remove temporary columns
df = df.drop("id", "default_probability")

print("Dataset successfully generated!")


Dataset successfully generated!


In [0]:

# Display sample records
display(df.limit(10))


loan_id,age,annual_income,credit_score,loan_amount,loan_purpose,employment_type,debt_to_income,interest_rate,default_flag,loan_status
LN0,64,94470,838,13197.17304926501,Education,Full-time,0.08,5.42,0,Non-Default
LN1,24,142870,332,44631.31636605435,Personal,Self-employed,0.33,23.13,0,Non-Default
LN2,25,68596,578,23667.077416877008,Education,Full-time,0.16,14.52,1,Default
LN3,46,66172,539,39637.60002653321,Education,Self-employed,0.17,15.89,0,Non-Default
LN4,57,95967,490,4202.061354638406,Personal,Full-time,0.28,17.6,0,Non-Default
LN5,55,120167,335,13382.429760097779,Home Improvement,Full-time,0.02,23.03,0,Non-Default
LN6,46,21136,529,2269.145558028822,Education,Self-employed,0.3,16.24,0,Non-Default
LN7,51,37378,769,46221.56989768104,Personal,Full-time,0.15,7.84,0,Non-Default
LN8,34,77454,427,32774.86402189692,Home Improvement,Self-employed,0.4,19.81,1,Default
LN9,55,47768,537,48649.30519989215,Personal,Full-time,0.51,15.96,0,Non-Default


In [0]:

# Dataset overview

print("Total records:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()


Total records: 10000
Total columns: 11
root
 |-- loan_id: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- loan_amount: double (nullable = false)
 |-- loan_purpose: string (nullable = false)
 |-- employment_type: string (nullable = false)
 |-- debt_to_income: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- default_flag: integer (nullable = false)
 |-- loan_status: string (nullable = false)



In [0]:

# BRONZE LAYER - Raw Data Storage

bronze_table = "default.credit_risk_bronze"

(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Bronze table created successfully: {bronze_table}")


Bronze table created successfully: default.credit_risk_bronze


In [0]:

# Read Bronze table

df_bronze = spark.table("default.credit_risk_bronze")

print("Bronze records:", df_bronze.count())

display(df_bronze.limit(5))


Bronze records: 10000


loan_id,age,annual_income,credit_score,loan_amount,loan_purpose,employment_type,debt_to_income,interest_rate,default_flag,loan_status
LN0,64,94470,838,13197.17304926501,Education,Full-time,0.08,5.42,0,Non-Default
LN1,24,142870,332,44631.31636605435,Personal,Self-employed,0.33,23.13,0,Non-Default
LN2,25,68596,578,23667.077416877008,Education,Full-time,0.16,14.52,1,Default
LN3,46,66172,539,39637.60002653321,Education,Self-employed,0.17,15.89,0,Non-Default
LN4,57,95967,490,4202.061354638406,Personal,Full-time,0.28,17.6,0,Non-Default


In [0]:
%sql

SELECT
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(),
        2
    ) AS percentage
FROM default.credit_risk_bronze
GROUP BY loan_status
ORDER BY total_loans DESC;


loan_status,total_loans,percentage
Non-Default,7892,78.92
Default,2108,21.08


In [0]:
%sql

SELECT
    CASE
        WHEN credit_score < 580 THEN 'Low Score'
        WHEN credit_score < 670 THEN 'Medium Score'
        ELSE 'High Score'
    END AS credit_score_category,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS default_loans,

    ROUND(AVG(default_flag) * 100, 2) AS default_rate

FROM default.credit_risk_bronze

GROUP BY 1
ORDER BY default_rate DESC;


credit_score_category,total_loans,default_loans,default_rate
Low Score,5108,1554,30.42
Medium Score,1607,304,18.92
High Score,3285,250,7.61


In [0]:
%sql

SELECT
    loan_status,
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amount), 2) AS total_loan_amount,
    ROUND(AVG(loan_amount), 2) AS average_loan_amount

FROM default.credit_risk_bronze

GROUP BY loan_status;


loan_status,total_loans,total_loan_amount,average_loan_amount
Non-Default,7892,2.0377057836E8,25819.89
Default,2108,5.273223952E7,25015.29


In [0]:

# SILVER LAYER - Data Cleaning and Validation

# Read Bronze table
df_bronze = spark.table("default.credit_risk_bronze")

# Remove duplicate loan records
df_silver = df_bronze.dropDuplicates(["loan_id"])

# Remove records with missing critical information
df_silver = df_silver.dropna(
    subset=[
        "loan_id",
        "age",
        "annual_income",
        "credit_score",
        "loan_amount",
        "debt_to_income",
        "default_flag"
    ]
)

# Apply business validation rules
df_silver = df_silver.filter(
    (F.col("age").between(18, 100)) &
    (F.col("annual_income") > 0) &
    (F.col("credit_score").between(300, 850)) &
    (F.col("loan_amount") > 0) &
    (F.col("debt_to_income").between(0, 1)) &
    (F.col("default_flag").isin(0, 1))
)

# Create analytical categories
df_silver = (
    df_silver
    .withColumn(
        "credit_score_category",
        F.when(F.col("credit_score") < 580, "Low Score")
         .when(F.col("credit_score") < 670, "Medium Score")
         .otherwise("High Score")
    )
    .withColumn(
        "income_category",
        F.when(F.col("annual_income") < 40000, "Low Income")
         .when(F.col("annual_income") < 80000, "Medium Income")
         .otherwise("High Income")
    )
    .withColumn(
        "loan_amount_category",
        F.when(F.col("loan_amount") < 10000, "Small Loan")
         .when(F.col("loan_amount") < 30000, "Medium Loan")
         .otherwise("Large Loan")
    )
)

print("Silver transformation completed!")


Silver transformation completed!


In [0]:

# Data Quality Report

total_records = df_silver.count()

null_records = df_silver.filter(
    F.col("loan_id").isNull() |
    F.col("annual_income").isNull() |
    F.col("credit_score").isNull()
).count()

duplicate_records = (
    df_silver.count() -
    df_silver.select("loan_id").distinct().count()
)

print("Total records:", total_records)
print("Null records:", null_records)
print("Duplicate records:", duplicate_records)
print("Total columns:", len(df_silver.columns))


Total records: 10000
Null records: 0
Duplicate records: 0
Total columns: 14


In [0]:

# Save cleaned data as Delta table

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("default.credit_risk_silver")
)

print("Silver table created successfully!")


Silver table created successfully!


In [0]:
%sql

SELECT
    credit_score_category,
    income_category,
    COUNT(*) AS total_customers,
    ROUND(AVG(default_flag) * 100, 2) AS default_rate
FROM default.credit_risk_silver
GROUP BY
    credit_score_category,
    income_category
ORDER BY default_rate DESC;


credit_score_category,income_category,total_customers,default_rate
Low Score,High Income,2740,31.02
Low Score,Low Income,802,30.3
Low Score,Medium Income,1566,29.44
Medium Score,Low Income,244,22.54
Medium Score,Medium Income,491,19.14
Medium Score,High Income,872,17.78
High Score,Low Income,525,8.19
High Score,Medium Income,1020,7.55
High Score,High Income,1740,7.47


In [0]:
%sql

CREATE OR REPLACE TABLE default.credit_risk_gold_kpis AS

SELECT
    COUNT(*) AS total_loans,

    SUM(default_flag) AS default_loans,

    ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct,

    ROUND(SUM(loan_amount), 2) AS total_loan_amount,

    ROUND(
        SUM(CASE
            WHEN default_flag = 1 THEN loan_amount
            ELSE 0
        END), 2
    ) AS default_exposure,

    ROUND(AVG(loan_amount), 2) AS average_loan_amount,

    ROUND(
        AVG(CASE
            WHEN default_flag = 1 THEN loan_amount
        END), 2
    ) AS average_default_loan_amount

FROM default.credit_risk_silver;


num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE default.credit_risk_gold_score AS

SELECT
    credit_score_category,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS default_loans,

    ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct,

    ROUND(SUM(loan_amount), 2) AS total_loan_amount,

    ROUND(
        SUM(CASE
            WHEN default_flag = 1 THEN loan_amount
            ELSE 0
        END), 2
    ) AS default_exposure

FROM default.credit_risk_silver

GROUP BY credit_score_category;


num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE default.credit_risk_gold_purpose AS

SELECT
    loan_purpose,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS default_loans,

    ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct,

    ROUND(SUM(loan_amount), 2) AS total_loan_amount,

    ROUND(
        SUM(CASE
            WHEN default_flag = 1 THEN loan_amount
            ELSE 0
        END), 2
    ) AS default_exposure

FROM default.credit_risk_silver

GROUP BY loan_purpose;


num_affected_rows,num_inserted_rows


In [0]:
%sql

CREATE OR REPLACE TABLE default.credit_risk_gold_income AS

SELECT
    income_category,

    COUNT(*) AS total_loans,

    SUM(default_flag) AS default_loans,

    ROUND(AVG(default_flag) * 100, 2) AS default_rate_pct,

    ROUND(SUM(loan_amount), 2) AS total_loan_amount,

    ROUND(
        SUM(CASE
            WHEN default_flag = 1 THEN loan_amount
            ELSE 0
        END), 2
    ) AS default_exposure

FROM default.credit_risk_silver

GROUP BY income_category;


num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT * FROM default.credit_risk_gold_kpis;


total_loans,default_loans,default_rate_pct,total_loan_amount,default_exposure,average_loan_amount,average_default_loan_amount
10000,2108,21.08,2.5650281788E8,5.273223952E7,25650.28,25015.29


In [0]:
%sql
SELECT * FROM default.credit_risk_gold_score;


credit_score_category,total_loans,default_loans,default_rate_pct,total_loan_amount,default_exposure
High Score,3285,250,7.61,8.428443483E7,6341579.68
Medium Score,1607,304,18.92,4.135147595E7,7261054.12
Low Score,5108,1554,30.42,1.308669071E8,3.912960572E7


In [0]:
%sql
SELECT * FROM default.credit_risk_gold_purpose;


loan_purpose,total_loans,default_loans,default_rate_pct,total_loan_amount,default_exposure
Education,3688,735,19.93,9.470656799E7,1.817211028E7
Debt Consolidation,967,206,21.3,2.477795349E7,4943435.62
Personal,2522,542,21.49,6.524938682E7,1.370222941E7
Home Improvement,2823,625,22.14,7.176890959E7,1.59144642E7


In [0]:
%sql
SELECT * FROM default.credit_risk_gold_income;

income_category,total_loans,default_loans,default_rate_pct,total_loan_amount,default_exposure
Low Income,1571,341,21.71,4.130182865E7,8833364.79
High Income,5352,1135,21.21,1.3736871958E8,2.832859438E7
Medium Income,3077,632,20.54,7.783226965E7,1.557028035E7


In [0]:
%sql

SELECT
    COUNT(*) AS total_loans,

    SUM(default_flag) AS default_loans,

    ROUND(AVG(default_flag) * 100, 2)
        AS default_rate_pct,

    ROUND(SUM(loan_amount), 2)
        AS total_loan_amount,

    ROUND(
        SUM(
            CASE
                WHEN default_flag = 1
                THEN loan_amount
                ELSE 0
            END
        ), 2
    ) AS default_exposure

FROM default.credit_risk_silver;


total_loans,default_loans,default_rate_pct,total_loan_amount,default_exposure
10000,2108,21.08,2.5650281788E8,5.273223952E7


In [0]:
%sql
SELECT
    credit_score_category,
    ROUND(SUM(loan_amount), 2) AS total_loan_amount,
    ROUND(
        SUM(CASE
            WHEN default_flag = 1 THEN loan_amount
            ELSE 0
        END), 2
    ) AS default_exposure,
    ROUND(
        SUM(CASE
            WHEN default_flag = 1 THEN loan_amount
            ELSE 0
        END) / SUM(loan_amount) * 100, 2
    ) AS default_exposure_pct
FROM default.credit_risk_silver
GROUP BY credit_score_category
ORDER BY credit_score_category;

credit_score_category,total_loan_amount,default_exposure,default_exposure_pct
High Score,8.428443483E7,6341579.68,7.52
Low Score,1.308669071E8,3.912960572E7,29.9
Medium Score,4.135147595E7,7261054.12,17.56
